# Prueba de rachas (Runs test) – Independencia (Up/Down)

**Materia:** Simulación  
**Objetivo:** verificar si una secuencia de números (pseudo)aleatorios es independiente.

## Hipótesis
- **H0:** los números {xᵢ} son independientes.
- **H1:** los números {xᵢ} no son independientes.

## Procedimiento (Up/Down)
1. Construir la secuencia binaria **S**:
   - Sᵢ = 0 si xᵢ ≤ xᵢ₋₁
   - Sᵢ = 1 si xᵢ > xᵢ₋₁
2. Contar el número de rachas observadas **C₀** en S.
3. Calcular:
   - μ = (2n − 1) / 3  
   - σ² = (16n − 29) / 90  
   - Z₀ = (C₀ − μ) / σ
4. Decisión (bilateral, nivel α):
   - Aceptar H0 si |Z₀| ≤ z_{α/2}

In [1]:
import math
import pandas as pd

# scipy가 있으면 사용(더 짧고 정확)
try:
    from scipy.stats import norm
    SCIPY_OK = True
except Exception:
    SCIPY_OK = False

def runs_test_updown(x, alpha=0.05):
    n = len(x)
    if n < 2:
        raise ValueError("Se requiere n >= 2.")

    # S: 0 if down/same, 1 if up
    S = [1 if x[i] > x[i-1] else 0 for i in range(1, n)]

    # C0: # of runs in S
    C0 = 1 + sum(S[i] != S[i-1] for i in range(1, len(S)))

    mu = (2*n - 1) / 3
    var = (16*n - 29) / 90
    sigma = math.sqrt(var)
    Z0 = (C0 - mu) / sigma

    if SCIPY_OK:
        zcrit = norm.ppf(1 - alpha/2)
        pval = 2 * (1 - norm.cdf(abs(Z0)))
    else:
        # fallback: N(0,1) via erf
        cdf = lambda z: 0.5 * (1 + math.erf(z / math.sqrt(2)))
        # zcrit by binary search
        target = 1 - alpha/2
        lo, hi = -10, 10
        for _ in range(80):
            mid = (lo + hi) / 2
            lo, hi = (mid, hi) if cdf(mid) < target else (lo, mid)
        zcrit = (lo + hi) / 2
        pval = 2 * (1 - cdf(abs(Z0)))

    decision = "No se rechaza H0 (independientes)" if abs(Z0) <= zcrit else "Se rechaza H0 (no independientes)"

    # 결과 표(깔끔)
    df = pd.DataFrame([{
        "n": n,
        "C0 (rachas)": C0,
        "μ": mu,
        "σ²": var,
        "σ": sigma,
        "Z0": Z0,
        "z_(α/2)": zcrit,
        "p-value": pval,
        "Decisión": decision
    }])

    return df, S

In [3]:
# ✅ Pega aquí tu lista de números (misma muestra de la tarea anterior)
x = [
    [0.78961, 0.05230, 0.10699, 0.55877, 0.14151],
    [0.76086, 0.12079, 0.27738, 0.65726, 0.79269],
    [0.80548, 0.82654, 0.29453, 0.20852, 0.42989],
    [0.58518, 0.98611, 0.34488, 0.34358, 0.11537],
    [0.89898, 0.57880, 0.67621, 0.05010, 0.00121],
    [0.28269, 0.73059, 0.70119, 0.18284, 0.49962],
    [0.38618, 0.76910, 0.68334, 0.55170, 0.10850],
    [0.79982, 0.45679, 0.21631, 0.87616, 0.55743],
    [0.58962, 0.33216, 0.03185, 0.61168, 0.09264],
    [0.69623, 0.17028, 0.05475, 0.91512, 0.76262],
    [0.29931, 0.30861, 0.83358, 0.51781, 0.03272],
    [0.57410, 0.26593, 0.85903, 0.43338, 0.35286],
    [0.24000, 0.65559, 0.38507, 0.90829, 0.94187],
    [0.93655, 0.88809, 0.81772, 0.36982, 0.19904],
    [0.54325, 0.62400, 0.09133, 0.41678, 0.33954],
    [0.58244, 0.85853, 0.88752, 0.33729, 0.15506],
    [0.23949, 0.53559, 0.33381, 0.49383, 0.75103],
    [0.19962, 0.65002, 0.74579, 0.79113, 0.63453],
    [0.19147, 0.40644, 0.08128, 0.78435, 0.22724],
    [0.22287, 0.07281, 0.64183, 0.44267, 0.72102],
]



alpha = 0.05
tabla_resultados, S = runs_test_updown(x, alpha=alpha)

# print
display(tabla_resultados.style.format({
    "μ": "{:.4f}", "σ²": "{:.4f}", "σ": "{:.4f}", "Z0": "{:.4f}",
    "z_(α/2)": "{:.4f}", "p-value": "{:.6f}"
}).hide(axis="index"))

print("S (primeros 40):", S[:40])

n,C0 (rachas),μ,σ²,σ,Z0,z_(α/2),p-value,Decisión
20,16,13.0000,3.2333,1.7981,1.6684,1.9600,0.095240,No se rechaza H0 (independientes)


S (primeros 40): [0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1]


## Conclusión

Con un nivel de significancia α = 0.05 se obtuvo:

- Z₀ = 1.668
- z_(α/2) = 1.96

Como |Z₀| < z_(α/2), **no se rechaza la hipótesis nula H₀**.

Por lo tanto, **la secuencia de números puede considerarse independiente**, lo que es consistente con el comportamiento esperado de números pseudoaleatorios.

## Interpretación

La prueba de rachas evalúa si los números de una secuencia presentan independencia.

El número de rachas observado fue cercano al valor esperado bajo independencia.  
El estadístico Z₀ se encuentra dentro del intervalo de aceptación.

Esto indica que **no existe evidencia estadística para afirmar que la secuencia no sea independiente**.